# CTR + Calibration (Ad-Tech Core)

## Beginner-friendly summary
This notebook builds click-through-rate models, compares model families, and shows why calibration matters for probability-based bidding decisions.

### What this notebook does
- Loads a real-world ad dataset sample
- Trains Logistic Regression and XGBoost models
- Evaluates ranking and probability metrics (AUC, log loss, etc.)
- Calibrates probabilities and visualizes calibration quality
- Simulates simple bidding impact based on predicted probability

### Major steps (in order)
1. Setup and data loading
2. Feature preprocessing
3. Logistic baseline
4. XGBoost model
5. Metric comparison
6. Calibration (before vs after)
7. Bidding-impact simulation
8. Key takeaways

### Side notes for beginners
- CTR models are often used to estimate click probability.
- Log loss strongly penalizes confident wrong predictions.
- Calibration curve near the 45-degree line means predicted probabilities better match observed outcomes.


## 1. Setup

> Side note: this cell loads libraries, defines local paths, and sets runtime knobs (`CTR_MAX_ROWS`, `CTR_MAX_FEATURES`, `RUN_XGBOOST`).

In [ ]:
# Side note (beginner):
# 1) Import modeling and plotting libraries.
# 2) Define data paths.
# 3) Set small defaults so notebook runs reliably.

from __future__ import annotations

import os
import zipfile
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_svmlight_file
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

from xgboost import XGBClassifier

sns.set_theme(style='whitegrid')
np.random.seed(42)

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Avazu_x1 preprocessed split from RecZoo tooling (public internet source)
AVAZU_URL = 'https://worksheets.codalab.org/rest/bundles/0xf5ab597052744680b1a55986557472c7/contents/blob/'
ZIP_PATH = DATA_DIR / 'avazu_x1.zip'
EXTRACT_DIR = DATA_DIR / 'avazu_x1'

MAX_ROWS = int(os.getenv('CTR_MAX_ROWS', '15000'))
MAX_FEATURES = int(os.getenv('CTR_MAX_FEATURES', '50000'))
RUN_XGBOOST = os.getenv('RUN_XGBOOST', '1').lower() in {'1', 'true', 'yes'}
RANDOM_STATE = 42

## 2. Data Source + Load

### Data provenance
- Dataset family: **Avazu CTR prediction** (real-world mobile ad click logs).
- Download source used in this notebook:
  `https://worksheets.codalab.org/rest/bundles/0xf5ab597052744680b1a55986557472c7/contents/blob/`
- Origin reference: RecZoo Avazu preprocessing scripts.
- Local files written to: `data/avazu_x1.zip` and `data/avazu_x1/`.

> Beginner side note: this section answers “where data comes from” and “what local files should exist after setup.”

> Beginner note: the full dataset is large. We use sampled data (`CTR_MAX_ROWS`, default 15k rows) for fast iteration.

In [ ]:
# Side note (beginner):
# 1) Reuse local files when present.
# 2) Extract libsvm files.
# 3) Create sampled libsvm first (do not load full file into memory).
# 4) Load sample and print quick sanity metrics.

def download_file(url: str, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f'Using existing file: {out_path}')
        return
    print(f'Downloading dataset zip to {out_path} ...')
    urllib.request.urlretrieve(url, out_path)
    print('Download complete.')


def extract_zip(zip_path: Path, extract_dir: Path) -> None:
    if extract_dir.exists() and any(extract_dir.rglob('*.libsvm')):
        print(f'Using existing extracted data in: {extract_dir}')
        return
    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {zip_path} -> {extract_dir} ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    print('Extraction complete.')


# Download is disabled by default for reruns.
# Uncomment once if local data is missing.
# download_file(AVAZU_URL, ZIP_PATH)

if not ZIP_PATH.exists() and not EXTRACT_DIR.exists():
    raise FileNotFoundError(
        'Avazu data not found locally. Uncomment download_file(...) once to fetch it.'
    )

extract_zip(ZIP_PATH, EXTRACT_DIR)

libsvm_candidates = sorted(EXTRACT_DIR.rglob('*.libsvm'))
if not libsvm_candidates:
    raise FileNotFoundError(f'No .libsvm files found under {EXTRACT_DIR}')

data_path = next((p for p in libsvm_candidates if 'train' in p.name.lower()), libsvm_candidates[0])
print(f'Using dataset file: {data_path}')

sample_path = DATA_DIR / f'avazu_x1_sample_{MAX_ROWS}.libsvm'
if not sample_path.exists():
    print(f'Creating sampled libsvm file: {sample_path}')
    with open(data_path, 'r', encoding='utf-8', errors='ignore') as src, open(sample_path, 'w', encoding='utf-8') as dst:
        for i, line in enumerate(src):
            if i >= MAX_ROWS:
                break
            dst.write(line)

X_sparse, y_raw = load_svmlight_file(str(sample_path))
X = X_sparse[:, :min(MAX_FEATURES, X_sparse.shape[1])]
y = pd.Series(y_raw).astype(int)

print({'rows_used': int(X.shape[0]), 'features': int(X.shape[1]), 'target': 'click', 'ctr_mean': round(float(y.mean()), 5)})

## 3. Train Logistic + XGBoost

> Side note: we always train Logistic Regression baseline; we train XGBoost when `RUN_XGBOOST=1`, calibrate it, and compare all models in one table.

In [ ]:
# Side note (beginner):
# 1) Split train/test.
# 2) Train Logistic Regression baseline.
# 3) Optionally train + calibrate XGBoost.
# 4) Compute log loss / Brier / AUC / ECE for each model.

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y.values, test_size=0.2, random_state=RANDOM_STATE, stratify=y.values
)

lr_model = LogisticRegression(max_iter=800, class_weight='balanced', n_jobs=1)
lr_model.fit(X_train_full, y_train_full)
p_lr = lr_model.predict_proba(X_test)[:, 1]

all_predictions = {'Logistic Regression': p_lr}

if RUN_XGBOOST:
    X_train, X_cal, y_train, y_cal = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=y_train_full,
    )

    # Compact latent features improve stability on constrained environments.
    svd = TruncatedSVD(n_components=32, random_state=RANDOM_STATE)
    X_train_svd = svd.fit_transform(X_train)
    X_cal_svd = svd.transform(X_cal)
    X_test_svd = svd.transform(X_test)

    xgb_model = XGBClassifier(
        n_estimators=60,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        eval_metric='logloss',
        tree_method='hist',
        n_jobs=1,
        random_state=RANDOM_STATE,
    )
    xgb_model.fit(X_train_svd, y_train)
    p_xgb = xgb_model.predict_proba(X_test_svd)[:, 1]

    # Fast probability calibration on held-out split.
    xgb_cal = CalibratedClassifierCV(estimator=xgb_model, method='sigmoid', cv='prefit')
    xgb_cal.fit(X_cal_svd, y_cal)
    p_xgb_cal = xgb_cal.predict_proba(X_test_svd)[:, 1]

    all_predictions['XGBoost (raw)'] = p_xgb
    all_predictions['XGBoost (calibrated)'] = p_xgb_cal


def ece_score(y_true: np.ndarray, p: np.ndarray, bins: int = 15) -> float:
    y_true = np.asarray(y_true)
    p = np.asarray(p)
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for i in range(bins):
        m = (p >= edges[i]) & (p < edges[i + 1])
        if m.sum() == 0:
            continue
        conf = p[m].mean()
        acc = y_true[m].mean()
        ece += (m.sum() / len(p)) * abs(acc - conf)
    return float(ece)


metrics = pd.DataFrame([
    {
        'model': name,
        'log_loss': log_loss(y_test, probs),
        'brier': brier_score_loss(y_test, probs),
        'roc_auc': roc_auc_score(y_test, probs),
        'ece': ece_score(y_test, probs),
    }
    for name, probs in all_predictions.items()
])

metrics = metrics.sort_values('log_loss').reset_index(drop=True)
metrics.style.format({'log_loss': '{:.5f}', 'brier': '{:.5f}', 'roc_auc': '{:.5f}', 'ece': '{:.5f}'})

## 4. Calibration Plot

> Beginner note: a perfectly calibrated model lies on the diagonal. If predicted 0.20 CTR, roughly 20% of those impressions should click.

> Side note: the 45° line is `y = x` (predicted probability equals observed click rate). Above line = under-confident model; below line = over-confident model.

In [ ]:
# Side note (beginner): this chart compares each model's calibration curve against perfect calibration.

plt.figure(figsize=(7, 6))

for name, probs in all_predictions.items():
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=12, strategy='quantile')
    plt.plot(mean_pred, frac_pos, marker='o', label=name)

plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
plt.xlabel('Predicted CTR probability')
plt.ylabel('Observed click rate')
plt.title('CTR Reliability / Calibration Curve')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Diagnostics: Log Loss + Learning Curve

> Side note: log loss checks probability quality; learning curves show whether more data may improve generalization.

In [ ]:
# Side note (beginner): lower log loss is better.
# If train loss is much lower than validation loss, model may be overfitting.

from sklearn.model_selection import learning_curve
from sklearn.base import clone

if 'metrics' in globals() and not metrics.empty:
    plt.figure(figsize=(9, 4))
    order = metrics.sort_values('log_loss', ascending=True)
    sns.barplot(data=order, x='model', y='log_loss', palette='magma')
    plt.title('CTR Model Log Loss Comparison')
    plt.xlabel('Model')
    plt.ylabel('Log loss (lower is better)')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()
    display(order[['model', 'log_loss', 'brier', 'roc_auc', 'ece']].reset_index(drop=True))
else:
    print('Metrics table not found. Run training cell first.')

if all(v in globals() for v in ['lr_model', 'X_train_full', 'y_train_full']):
    max_curve_rows = min(30000, X_train_full.shape[0])
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.choice(X_train_full.shape[0], size=max_curve_rows, replace=False)
    X_lc = X_train_full[idx]
    y_lc = np.asarray(y_train_full)[idx]

    sizes, train_scores, valid_scores = learning_curve(
        estimator=clone(lr_model),
        X=X_lc,
        y=y_lc,
        cv=3,
        scoring='neg_log_loss',
        train_sizes=np.linspace(0.1, 1.0, 5),
        n_jobs=1,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    train_loss = -train_scores.mean(axis=1)
    valid_loss = -valid_scores.mean(axis=1)

    plt.figure(figsize=(8, 4))
    plt.plot(sizes, train_loss, marker='o', label='Train log loss')
    plt.plot(sizes, valid_loss, marker='o', label='Validation log loss')
    plt.title('Learning Curve (CTR Logistic Regression, log loss)')
    plt.xlabel('Training examples used')
    plt.ylabel('Log loss (lower is better)')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Learning curve skipped: lr_model / X_train_full / y_train_full not found.')


## 6. Bidding Impact Simulation

We simulate a simple decision rule:

$$
\text{value per click }(vpc)=2.00,\qquad \text{impression cost }(cpi)=0.01
$$

$$
\text{Bid if } \hat{p}_{\text{click}} \cdot vpc > cpi
$$

> Currency note: these are in dollars (USD): `vpc = \$2.00`, `cpi = \$0.01`.

> Side note: this turns model probabilities into an actual decision policy (buy vs skip impression).

This approximates how calibration affects spend efficiency in auctions.

In [ ]:
# Side note (beginner): this cell computes business KPIs from a simple bidding rule.

vpc = 2.00
cpi = 0.01


def bidding_report(name: str, probs: np.ndarray, y_true: pd.Series, value_per_click: float, cost_per_impression: float) -> dict:
    take = (probs * value_per_click) > cost_per_impression
    selected = int(take.sum())
    if selected == 0:
        return {
            'model': name,
            'impressions_bought': 0,
            'buy_rate': 0.0,
            'realized_ctr': 0.0,
            'revenue': 0.0,
            'cost': 0.0,
            'profit': 0.0,
        }

    y_sel = y_true[take]
    clicks = int(y_sel.sum())
    revenue = clicks * value_per_click
    cost = selected * cost_per_impression
    profit = revenue - cost
    return {
        'model': name,
        'impressions_bought': selected,
        'buy_rate': selected / len(y_true),
        'realized_ctr': float(y_sel.mean()),
        'revenue': revenue,
        'cost': cost,
        'profit': profit,
    }


bid_df = pd.DataFrame([
    bidding_report(name, probs, pd.Series(y_test).reset_index(drop=True), vpc, cpi)
    for name, probs in all_predictions.items()
]).sort_values('profit', ascending=False)

bid_df.style.format({
    'buy_rate': '{:.2%}',
    'realized_ctr': '{:.4f}',
    'revenue': '${:.2f}',
    'cost': '${:.2f}',
    'profit': '${:.2f}',
})

## 7. Key Learning

- In CTR systems, optimize not just ROC-AUC but also **log loss** and **calibration quality**.
- Better-calibrated probabilities reduce under/over-bidding risk.
- Bidding impact should be evaluated with business assumptions (value-per-click, impression cost, auction behavior).

> Side note: this is the final summary cell that translates model metrics into business decision impact.

> Next step: evaluate by traffic segments (device, hour, campaign) and monitor calibration drift over time.